# Fig. 22 SRDM TF-Screening Diagnostic

This notebook uses dedicated high-statistics DaMaSCUS-SUN Fig. 22 flux files, when present, to generate a DMeRates silicon SRDM diagnostic with the **current Thomas-Fermi screening path**.

Important scope note: this is not a true Fig. 22 Lindhard/dielectric reproduction. Fig. 22 labels screened curves as `1/|epsilon|^2`; this notebook intentionally leaves screening alone and uses DMeRates' existing `DoScreen=True` Thomas-Fermi convention for QEDark/QCDark1.

Expected external flux directory:

`/Users/ansh/Local/SENSEI/fig22_validation/`

Required flux files can either be the flat copied files or the DaMaSCUS run outputs:

- `srdm_dphidv_DPLM_fig22_mchi_10keV_sigmae_1e-35.txt` or `fig22_DPLM_mchi_10keV_sigmae_1e-35_avg/Differential_SRDM_Flux.txt`
- `srdm_dphidv_DPLM_fig22_mchi_100keV_sigmae_1e-35.txt` or `fig22_DPLM_mchi_100keV_sigmae_1e-35_avg/Differential_SRDM_Flux.txt`
- `srdm_dphidv_DPLM_fig22_mchi_1MeV_sigmae_1e-35.txt` or `fig22_DPLM_mchi_1MeV_sigmae_1e-35_avg/Differential_SRDM_Flux.txt`

Each file must have two columns: velocity in `km/s`, then `dPhi/dv` in `cm^-2 s^-1 (km/s)^-1`.


In [ ]:
import json
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numericalunits as nu
import torch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from DMeRates.DMeRate import DMeRate
from DMeRates.data.registry import DataRegistry

EXTERNAL_FLUX_DIR = Path('/Users/ansh/Local/SENSEI/fig22_validation')
LOCAL_SRDM_DIR = REPO_ROOT / 'halo_data' / 'srdm'
MANIFEST_PATH = DataRegistry.srdm_manifest()

FIG22_FLUX_SPECS = [
    dict(
        label='10 keV',
        mX_eV=1.0e4,
        sigma_e_cm2=1.0e-35,
        filename='srdm_dphidv_DPLM_fig22_mchi_10keV_sigmae_1e-35.txt',
        generated_path='fig22_DPLM_mchi_10keV_sigmae_1e-35_avg/Differential_SRDM_Flux.txt',
        config='config_fig22_DPLM_mchi_10keV_sigmae_1e-35.cfg',
    ),
    dict(
        label='100 keV',
        mX_eV=1.0e5,
        sigma_e_cm2=1.0e-35,
        filename='srdm_dphidv_DPLM_fig22_mchi_100keV_sigmae_1e-35.txt',
        generated_path='fig22_DPLM_mchi_100keV_sigmae_1e-35_avg/Differential_SRDM_Flux.txt',
        config='config_fig22_DPLM_mchi_100keV_sigmae_1e-35.cfg',
    ),
    dict(
        label='1 MeV',
        mX_eV=1.0e6,
        sigma_e_cm2=1.0e-35,
        filename='srdm_dphidv_DPLM_fig22_mchi_1MeV_sigmae_1e-35.txt',
        generated_path='fig22_DPLM_mchi_1MeV_sigmae_1e-35_avg/Differential_SRDM_Flux.txt',
        config='config_fig22_DPLM_mchi_1MeV_sigmae_1e-35.cfg',
    ),
]

plt.rcParams.update({
    'figure.figsize': (18, 4.8),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'font.size': 11,
})


In [ ]:
def validate_flux_file(path):
    data = np.loadtxt(path, comments='#')
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f'{path} must be a two-column flux file')
    v_kms = data[:, 0]
    dphi_dv = data[:, 1]
    if not np.all(np.isfinite(v_kms)) or not np.all(np.isfinite(dphi_dv)):
        raise ValueError(f'{path} contains non-finite values')
    if np.any(v_kms < 0.0):
        raise ValueError(f'{path} contains negative velocities')
    if np.any(dphi_dv < 0.0):
        raise ValueError(f'{path} contains negative flux values')
    return data


def install_and_register_fig22_fluxes():
    missing = []
    installed = []
    for spec in FIG22_FLUX_SPECS:
        src = EXTERNAL_FLUX_DIR / spec['filename']
        generated = EXTERNAL_FLUX_DIR / spec.get('generated_path', '')
        if not src.exists():
            src = generated
        if not src.exists():
            missing.append(EXTERNAL_FLUX_DIR / spec['filename'])
            missing.append(generated)
            continue
        validate_flux_file(src)
        dst = LOCAL_SRDM_DIR / spec['filename']
        if not dst.exists() or src.read_bytes() != dst.read_bytes():
            shutil.copyfile(src, dst)
        installed.append(dst)

    if missing:
        lines = '\n'.join(f'  - {p}' for p in missing)
        raise FileNotFoundError(
            'Missing Fig. 22 DaMaSCUS flux output files. Run/copy the DaMaSCUS results first:\n'
            f'{lines}'
        )

    manifest = json.loads(MANIFEST_PATH.read_text())
    files = manifest.setdefault('files', [])
    for spec in FIG22_FLUX_SPECS:
        entry = dict(
            mX_eV=float(spec['mX_eV']),
            sigma_e_cm2=float(spec['sigma_e_cm2']),
            FDMn=2,
            mediator_spin='vector',
            nominal_mX_eV=float(spec['mX_eV']),
            nominal_sigma_e_cm2=float(spec['sigma_e_cm2']),
            grid_index=None,
            grid_family='DPLM_fig22_damascus_sun',
            filename=spec['filename'],
            source='DaMaSCUS-SUN high-stat Fig. 22 validation rerun',
            url='https://github.com/temken/damascus-sun',
            upstream_filename='results/<ID>/Differential_SRDM_Flux.txt',
            upstream_config=spec['config'],
            retrieved='local',
            cross_section_convention='sigma_e_bar',
            flux_type='isotropic_angle_averaged',
            sample_size=100000,
            interpolation_points=1000,
            isoreflection_rings=1,
            use_medium_effects=True,
            screening_note='Flux includes solar in-medium effects from DaMaSCUS-SUN; detector screening below is DMeRates Thomas-Fermi.',
        )
        matches = [e for e in files if e.get('filename') == spec['filename']]
        if matches:
            existing = matches[0]
            for key in ('mX_eV', 'sigma_e_cm2', 'FDMn', 'mediator_spin'):
                if existing.get(key) != entry[key]:
                    raise RuntimeError(f'Conflicting manifest entry for {spec["filename"]}: {key}')
            existing.update(entry)
        else:
            files.append(entry)

    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + '\n')
    return installed

installed = install_and_register_fig22_fluxes()
print('Installed and registered Fig. 22 flux files:')
for path in installed:
    data = np.loadtxt(path, comments='#')
    print(f'  {path.name}: rows={len(data)}, v=[{data[:,0].min():.3g}, {data[:,0].max():.3g}] km/s, integral~{np.trapezoid(data[:,1], data[:,0]):.3e} cm^-2 s^-1')


In [ ]:
def run_public_drdE_curve(*, backend, spec, do_screen):
    dm = DMeRate('Si', form_factor_type=backend)
    rates, dRdE, prob = dm.calculate_rates(
        mX_array=[spec['mX_eV'] / 1.0e6],
        halo_model='srdm',
        FDMn=2,
        ne=[1],
        DoScreen=do_screen,
        sigma_e=spec['sigma_e_cm2'],
        mediator_spin='vector',
        debug=True,
    )
    E_eV = (dm.Earr / nu.eV).detach().cpu().numpy()
    y = (dRdE * nu.kg * nu.year * nu.eV).detach().cpu().numpy()
    return E_eV, y

all_curves = []
for spec in FIG22_FLUX_SPECS:
    curves = {}
    print(f"\nRunning {spec['label']} ({spec['mX_eV']:.6g} eV, sigma_e={spec['sigma_e_cm2']:.1e})")
    for label, backend, do_screen in [
        ('QEDark no TF', 'qedark', False),
        ('QEDark TF', 'qedark', True),
        ('QCDark1 no TF', 'qcdark', False),
        ('QCDark1 TF', 'qcdark', True),
    ]:
        E, y = run_public_drdE_curve(backend=backend, spec=spec, do_screen=do_screen)
        curves[label] = (E, y)
        print(f'  {label:14s}: finite={np.all(np.isfinite(y))}, positive bins={np.count_nonzero(y > 0)}, max={np.nanmax(y):.3e}')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    all_curves.append((spec, curves))


In [ ]:
styles = {
    'QEDark no TF': dict(color='#d62728', ls='--', lw=1.7),
    'QEDark TF': dict(color='#d62728', ls='-', lw=2.0),
    'QCDark1 no TF': dict(color='#ff9f1c', ls='--', lw=1.7),
    'QCDark1 TF': dict(color='#ff9f1c', ls='-', lw=2.0),
}

fig, axes = plt.subplots(1, len(all_curves), figsize=(18, 4.8), sharey=False)
if len(all_curves) == 1:
    axes = [axes]

for ax, (spec, curves) in zip(axes, all_curves):
    for label, (E, y) in curves.items():
        mask = (E >= 0.0) & (E <= 50.0) & np.isfinite(y) & (y > 0.0)
        ax.plot(E[mask], y[mask], label=label, **styles[label])
    ax.axvspan(15.0, 20.0, color='gold', alpha=0.12, label='15-20 eV' if ax is axes[0] else None)
    ax.set_yscale('log')
    ax.set_xlim(0.0, 50.0)
    ax.set_xlabel('Deposited energy E [eV]')
    ax.set_title(f"{spec['label']}, sigma_e=1e-35 cm^2")
    ax.text(0.98, 0.93, 'Si', transform=ax.transAxes, ha='right', va='top', color='0.45', fontsize=13)
axes[0].set_ylabel('dR/dE [1/eV/kg/year]')
axes[0].legend(loc='best', fontsize=9)
fig.suptitle('DMeRates SRDM diagnostic with DaMaSCUS Fig. 22 fluxes and Thomas-Fermi detector screening')
fig.tight_layout()
plt.show()


In [ ]:
# Mechanical sanity checks, not a Lindhard Fig. 22 acceptance test.
for spec, curves in all_curves:
    print(f"\nChecks for {spec['label']}")
    for family in ['QEDark', 'QCDark1']:
        _, y0 = curves[f'{family} no TF']
        _, y1 = curves[f'{family} TF']
        common = (y0 > 0.0) & (y1 > 0.0) & np.isfinite(y0) & np.isfinite(y1)
        if not np.any(common):
            raise AssertionError(f'{family}: no common positive finite bins')
        differs = not np.allclose(y0[common], y1[common])
        print(f'  {family:7s} TF differs from unscreened: {differs}')
        if not differs:
            raise AssertionError(f'{family} TF and unscreened spectra are numerically identical')

print('\nNOTE: These checks only validate notebook plumbing and TF-screened behavior.')
